# Notebook 7: Robustness and Sensitivity Analysis

## Purpose
Verify robustness of results.

## Tasks
- **Leave-one-author-out** validation
- **Bootstrap confidence intervals**
- **Alternative thresholds** (e.g., redefining "Top")

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import kruskal, mannwhitneyu
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 150
np.random.seed(42)

## 1. Load Data

In [ ]:
PROJECT_ROOT = Path().resolve().parent.parent.parent.parent
INPUT_FILE = PROJECT_ROOT / "results" / "stage10_correlation_analysis" / "statistical_analysis" / "indices_book.csv"
OUTPUT_DIR = PROJECT_ROOT / "results" / "stage10_correlation_analysis" / "statistical_analysis"

df = pd.read_csv(INPUT_FILE)
print(f"Loaded {len(df)} books")
if 'author_id' in df.columns:
    print(f"Authors: {df['author_id'].nunique()}")

## 2. Leave-One-Author-Out Validation

In [ ]:
def test_hypothesis_loao(data, index_col, group_col='group', author_col='author_id'):
    """Test hypothesis with leave-one-author-out validation."""
    if author_col not in data.columns:
        print(f"No {author_col} column found. Skipping LOAO.")
        return None
    
    authors = data[author_col].unique()
    results = []
    
    for author in tqdm(authors, desc="LOAO validation"):
        # Leave out this author
        subset = data[data[author_col] != author]
        
        if group_col in subset.columns:
            top_data = subset[subset[group_col] == 'Top'][index_col].dropna()
            trash_data = subset[subset[group_col] == 'Trash'][index_col].dropna()
            
            if len(top_data) > 0 and len(trash_data) > 0:
                u_stat, p_val = mannwhitneyu(top_data, trash_data, alternative='two-sided')
                
                # Effect size
                pooled_std = np.sqrt(((len(top_data) - 1) * top_data.std()**2 + 
                                    (len(trash_data) - 1) * trash_data.std()**2) / 
                                   (len(top_data) + len(trash_data) - 2))
                cohens_d = (top_data.mean() - trash_data.mean()) / pooled_std if pooled_std > 0 else 0
                
                results.append({
                    'excluded_author': author,
                    'p_value': p_val,
                    'cohens_d': cohens_d,
                    'mean_diff': top_data.mean() - trash_data.mean(),
                    'n_top': len(top_data),
                    'n_trash': len(trash_data)
                })
    
    return pd.DataFrame(results)

# Run LOAO for key indices
if 'author_id' in df.columns and 'group' in df.columns:
    key_indices = ['love_over_sex', 'hea_index', 'protective_minus_jealous', 'dark_vs_tender']
    available_indices = [idx for idx in key_indices if idx in df.columns]
    
    loao_results = {}
    for idx in available_indices:
        print(f"\nLOAO validation for {idx}:")
        result = test_hypothesis_loao(df, idx)
        if result is not None:
            loao_results[idx] = result
            print(f"  P-values range: {result['p_value'].min():.4f} - {result['p_value'].max():.4f}")
            print(f"  Significant (p<0.05) in {((result['p_value'] < 0.05).sum())} / {len(result)} iterations")
    
    # Save results
    for idx, result_df in loao_results.items():
        output_file = OUTPUT_DIR / f"loao_{idx}.csv"
        result_df.to_csv(output_file, index=False)
        print(f"✓ Saved: {output_file}")

## 3. Bootstrap Confidence Intervals

In [ ]:
def bootstrap_ci(data, statistic_func, n_bootstrap=1000, confidence=0.95, random_state=42):
    """Compute bootstrap confidence interval for a statistic."""
    np.random.seed(random_state)
    
    bootstrap_stats = []
    for _ in range(n_bootstrap):
        sample = np.random.choice(data, size=len(data), replace=True)
        stat = statistic_func(sample)
        bootstrap_stats.append(stat)
    
    alpha = 1 - confidence
    lower = np.percentile(bootstrap_stats, 100 * alpha / 2)
    upper = np.percentile(bootstrap_stats, 100 * (1 - alpha / 2))
    
    return {
        'statistic': statistic_func(data),
        'ci_lower': lower,
        'ci_upper': upper,
        'bootstrap_mean': np.mean(bootstrap_stats),
        'bootstrap_std': np.std(bootstrap_stats)
    }

# Bootstrap CIs for key statistics
if 'group' in df.columns:
    bootstrap_results = []
    
    for idx in available_indices:
        top_data = df[df['group'] == 'Top'][idx].dropna()
        trash_data = df[df['group'] == 'Trash'][idx].dropna()
        
        if len(top_data) > 0 and len(trash_data) > 0:
            # Bootstrap mean difference
            def mean_diff(sample):
                # This is simplified - in practice, you'd bootstrap both groups
                return np.mean(sample)
            
            top_ci = bootstrap_ci(top_data.values, np.mean)
            trash_ci = bootstrap_ci(trash_data.values, np.mean)
            
            bootstrap_results.append({
                'index': idx,
                'top_mean': top_ci['statistic'],
                'top_ci_lower': top_ci['ci_lower'],
                'top_ci_upper': top_ci['ci_upper'],
                'trash_mean': trash_ci['statistic'],
                'trash_ci_lower': trash_ci['ci_lower'],
                'trash_ci_upper': trash_ci['ci_upper'],
                'mean_diff': top_ci['statistic'] - trash_ci['statistic']
            })
    
    bootstrap_df = pd.DataFrame(bootstrap_results)
    print("\nBootstrap Confidence Intervals:")
    print(bootstrap_df)
    
    # Save
    output_file = OUTPUT_DIR / "bootstrap_confidence_intervals.csv"
    bootstrap_df.to_csv(output_file, index=False)
    print(f"\n✓ Saved: {output_file}")

## 4. Alternative Thresholds for "Top" Group

In [ ]:
# Test sensitivity to different definitions of "Top" group
if 'average_rating_weighted_mean' in df.columns:
    # Try different percentile thresholds
    thresholds = [0.75, 0.80, 0.85, 0.90, 0.95]  # Top 25%, 20%, 15%, 10%, 5%
    
    threshold_results = []
    
    for threshold in thresholds:
        # Define Top and Trash based on threshold
        top_percentile = (1 - threshold) * 100
        bottom_percentile = threshold * 100
        
        df_test = df.copy()
        df_test['group_alt'] = 'Mid'
        df_test.loc[df_test['average_rating_weighted_mean'] >= df_test['average_rating_weighted_mean'].quantile(threshold), 'group_alt'] = 'Top'
        df_test.loc[df_test['average_rating_weighted_mean'] <= df_test['average_rating_weighted_mean'].quantile(1 - threshold), 'group_alt'] = 'Trash'
        
        # Test hypotheses with alternative grouping
        for idx in available_indices:
            top_data = df_test[df_test['group_alt'] == 'Top'][idx].dropna()
            trash_data = df_test[df_test['group_alt'] == 'Trash'][idx].dropna()
            
            if len(top_data) > 0 and len(trash_data) > 0:
                u_stat, p_val = mannwhitneyu(top_data, trash_data, alternative='two-sided')
                
                threshold_results.append({
                    'threshold': threshold,
                    'top_percentile': top_percentile,
                    'index': idx,
                    'p_value': p_val,
                    'mean_diff': top_data.mean() - trash_data.mean(),
                    'n_top': len(top_data),
                    'n_trash': len(trash_data)
                })
    
    threshold_df = pd.DataFrame(threshold_results)
    print("\nSensitivity to Threshold Definition:")
    print(threshold_df)
    
    # Visualize
    if len(threshold_df) > 0:
        for idx in available_indices:
            idx_data = threshold_df[threshold_df['index'] == idx]
            if len(idx_data) > 0:
                plt.figure(figsize=(10, 6))
                plt.plot(idx_data['threshold'], idx_data['p_value'], marker='o')
                plt.axhline(0.05, color='r', linestyle='--', label='p=0.05')
                plt.xlabel('Threshold (Top Percentile)')
                plt.ylabel('P-value')
                plt.title(f'Sensitivity Analysis: {idx} P-value vs Threshold')
                plt.legend()
                plt.grid(True, alpha=0.3)
                plt.tight_layout()
                plt.savefig(OUTPUT_DIR / f'sensitivity_threshold_{idx}.png')
                plt.show()
    
    # Save
    output_file = OUTPUT_DIR / "sensitivity_threshold_results.csv"
    threshold_df.to_csv(output_file, index=False)
    print(f"\n✓ Saved: {output_file}")

## Summary

Robustness and sensitivity analysis complete. Next: Notebook 8 (Final Report Generation)